# LESSON 5.4: Back Projection
## Image Reconstruction from Projections

In this lesson:
- What is back projection (the "smearing" operation)
- Simple (unfiltered) back projection
- Why unfiltered back projection produces blurred images
- The $1/r$ blurring effect
- Building intuition for filtered back projection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import radon, iradon
from skimage.data import shepp_logan_phantom
from skimage.transform import rescale
from skimage.draw import disk, ellipse
from scipy.ndimage import rotate

## 1. The Back Projection Concept

**Back projection** is the inverse operation of projection:

- **Projection** (Radon Transform): Takes a 2-D image → produces 1-D projections (line integrals)
- **Back projection**: Takes 1-D projections → produces a 2-D image (by "smearing" each projection back)

### The Idea:
For each projection at angle $\theta$:
1. Take the 1-D projection values
2. "Smear" (replicate) each value along the direction of the original ray
3. Accumulate contributions from all angles

### Mathematical Definition:

$$\hat{f}(x, y) = \int_0^{\pi} g(x\cos\theta + y\sin\theta, \theta) \, d\theta$$

For each point $(x, y)$, we sum the projection values $g(\rho, \theta)$ at the corresponding $\rho = x\cos\theta + y\sin\theta$ over all angles.

In [ ]:
# Demonstrate back projection of a single projection
size = 128

# Create a simple point object
image = np.zeros((size, size))
rr, cc = disk((64, 64), 15)
image[rr, cc] = 1.0

def back_project_single(projection, angle_deg, size):
    """Back project a single projection: smear values along the ray direction."""
    # Create a 2D image by replicating the projection along columns
    bp = np.tile(projection.reshape(-1, 1), (1, size))
    # Crop or pad to match size
    if bp.shape[0] > size:
        start = (bp.shape[0] - size) // 2
        bp = bp[start:start+size, :]
    elif bp.shape[0] < size:
        padded = np.zeros((size, size))
        start = (size - bp.shape[0]) // 2
        padded[start:start+bp.shape[0], :] = bp
        bp = padded
    # Rotate to match the original projection angle
    bp = rotate(bp, -angle_deg, reshape=False, order=1)
    return bp

# Show back projection of single projections at different angles
angles_demo = [0, 45, 90, 135]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for i, angle in enumerate(angles_demo):
    # Forward projection
    projection = radon(image, theta=np.array([float(angle)]), circle=False)
    proj = projection[:, 0]
    
    # Back projection
    bp = back_project_single(proj, angle, size)
    
    # Original
    axes[0, i].imshow(image, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'Original', fontsize=11)
    
    # Projection
    axes[1, i].plot(proj, 'b-', linewidth=2)
    axes[1, i].set_title(f'Projection at {angle}°', fontsize=11)
    axes[1, i].grid(True, alpha=0.3)
    
    # Back projection
    axes[2, i].imshow(bp, cmap='hot')
    axes[2, i].set_title(f'Back Projected at {angle}°', fontsize=11)

plt.suptitle('Back Projection: "Smearing" a Projection Back Along Ray Direction',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Each back projection smears the 1D projection values along the ray direction.")
print("A single back projection gives NO localization — we need to combine many!")

## 2. Accumulating Back Projections

As we add more projections from different angles, the back projected image gradually converges to the original. Let's see how.

In [ ]:
# Show cumulative back projection with increasing number of angles
size = 128
image = np.zeros((size, size))
rr, cc = disk((64, 64), 20)
image[rr, cc] = 1.0
rr, cc = disk((45, 45), 8)
image[rr, cc] = 0.5

n_angles_list = [1, 2, 4, 10, 30, 90, 180, 360]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for idx, n_angles in enumerate(n_angles_list):
    angles = np.linspace(0, 180, n_angles, endpoint=False)
    
    # Compute sinogram
    sinogram = radon(image, theta=angles, circle=False)
    
    # Accumulate back projections
    reconstruction = np.zeros((size, size))
    for i, angle in enumerate(angles):
        bp = back_project_single(sinogram[:, i], angle, size)
        reconstruction += bp
    
    # Normalize
    if n_angles > 0:
        reconstruction /= n_angles
    
    axes[idx].imshow(reconstruction, cmap='gray')
    axes[idx].set_title(f'{n_angles} angle{"s" if n_angles > 1 else ""}', fontsize=11)
    axes[idx].axis('off')

plt.suptitle('Simple Back Projection: Accumulating Projections',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("With 1 angle: Just a smeared line.")
print("With 2 angles: Intersection begins to localize features.")
print("With many angles: Shape emerges but remains BLURRED.")
print("\nNote: Even with 360 angles, the result is blurred!")
print("This is the fundamental limitation of simple back projection.")

## 3. The $1/r$ Blurring Problem

Simple back projection produces a blurred version of the original image:

$$\hat{f}(x, y) = f(x, y) * \frac{1}{\sqrt{x^2 + y^2}} = f(x, y) * \frac{1}{r}$$

where $*$ denotes convolution and $r = \sqrt{x^2 + y^2}$.

### Why does this happen?
- Each point in the image contributes to projections at all angles
- When back projecting, energy is "smeared" radially outward
- The smearing follows a $1/r$ pattern (like a star artifact)

### In Fourier domain:
The $1/r$ blurring corresponds to a $1/|\omega|$ weighting in the frequency domain, which matches the density problem we discussed in Lesson 5.3.

In [ ]:
# Demonstrate the 1/r blurring
size = 128

# Create a point source
point = np.zeros((size, size))
point[64, 64] = 1.0

# Back project with many angles
n_angles = 360
angles = np.linspace(0, 180, n_angles, endpoint=False)
sinogram = radon(point, theta=angles, circle=False)

# Simple back projection using iradon without filter
bp_result = iradon(sinogram, theta=angles, filter_name=None, circle=False)

# Create theoretical 1/r PSF for comparison
y, x = np.mgrid[-size//2:size//2, -size//2:size//2]
r = np.sqrt(x**2 + y**2)
r[r == 0] = 0.5  # avoid division by zero
psf_1_over_r = 1.0 / r

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(point, cmap='gray')
axes[0].set_title('Point Source', fontsize=12)

im1 = axes[1].imshow(bp_result, cmap='hot')
axes[1].set_title('Back Projection of Point\n(Point Spread Function)', fontsize=11)
plt.colorbar(im1, ax=axes[1])

axes[2].imshow(psf_1_over_r, cmap='hot', vmax=5)
axes[2].set_title('Theoretical 1/r PSF', fontsize=12)

# Profile comparison
center = size // 2
profile_bp = bp_result[center, :]
profile_1r = psf_1_over_r[center, :]

x_pixels = np.arange(size) - center
axes[3].plot(x_pixels, profile_bp / profile_bp.max(), 'b-', linewidth=2,
            label='Back projection PSF')
axes[3].plot(x_pixels[x_pixels != 0],
            (1.0/np.abs(x_pixels[x_pixels != 0])) / (1.0/np.abs(x_pixels[x_pixels != 0])).max(),
            'r--', linewidth=2, label='Theoretical 1/|x|')
axes[3].set_title('PSF Profile (Horizontal)', fontsize=12)
axes[3].set_xlabel('Distance from center')
axes[3].set_ylabel('Normalized intensity')
axes[3].set_xlim([-50, 50])
axes[3].legend()
axes[3].grid(True, alpha=0.3)

plt.suptitle('The 1/r Blurring Problem in Simple Back Projection',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("A point source gets blurred into a 1/r pattern (star artifact).")
print("This means every feature in the reconstruction is surrounded by a halo.")
print("The solution: FILTER each projection before back projecting!")

## 4. Comparing Unfiltered Back Projection with Different Objects

Let's see how simple back projection performs on various test objects.

In [ ]:
# Back projection of various objects
size = 128
n_angles = 180
angles = np.linspace(0, 180, n_angles, endpoint=False)

# Object 1: Circle
obj1 = np.zeros((size, size))
rr, cc = disk((64, 64), 30)
obj1[rr, cc] = 1.0

# Object 2: Two circles
obj2 = np.zeros((size, size))
rr, cc = disk((50, 40), 15)
obj2[rr, cc] = 1.0
rr, cc = disk((80, 90), 15)
obj2[rr, cc] = 0.7

# Object 3: Shepp-Logan
obj3 = shepp_logan_phantom()
obj3 = rescale(obj3, size/400, anti_aliasing=True)
obj3 = obj3[:size, :size]

objects = [obj1, obj2, obj3]
names = ['Circle', 'Two Circles', 'Shepp-Logan']

fig, axes = plt.subplots(3, 3, figsize=(14, 14))

for i, (obj, name) in enumerate(zip(objects, names)):
    sinogram = radon(obj, theta=angles, circle=False)
    bp_unfiltered = iradon(sinogram, theta=angles, filter_name=None, circle=False)
    bp_filtered = iradon(sinogram, theta=angles, filter_name='ramp', circle=False)
    
    # Original
    axes[i, 0].imshow(obj, cmap='gray')
    axes[i, 0].set_title(f'Original: {name}', fontsize=11)
    
    # Unfiltered back projection
    axes[i, 1].imshow(bp_unfiltered, cmap='gray')
    axes[i, 1].set_title('Unfiltered Back Projection', fontsize=11)
    
    # Filtered back projection (preview)
    axes[i, 2].imshow(bp_filtered, cmap='gray')
    axes[i, 2].set_title('Filtered Back Projection', fontsize=11)

plt.suptitle('Unfiltered vs Filtered Back Projection Comparison',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Left: Original images")
print("Center: Simple back projection — severely blurred (1/r convolution)")
print("Right: Filtered back projection — much sharper (preview of Lesson 5.5)")

## 5. Step-by-Step Back Projection Animation

Let's watch how the reconstruction builds up as we add projections one by one.

In [ ]:
# Step-by-step back projection accumulation (Shepp-Logan)
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.25, anti_aliasing=True)
sz = phantom.shape[0]

n_angles = 180
angles = np.linspace(0, 180, n_angles, endpoint=False)
sinogram = radon(phantom, theta=angles, circle=True)

# Show reconstruction at selected stages
stages = [1, 5, 10, 20, 45, 90, 120, 180]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for idx, n in enumerate(stages):
    # Use only first n projections
    theta_subset = angles[:n]
    sino_subset = sinogram[:, :n]
    
    # Unfiltered back projection
    recon = iradon(sino_subset, theta=theta_subset, filter_name=None, circle=True)
    
    axes[idx].imshow(recon, cmap='gray')
    axes[idx].set_title(f'{n} projection{"s" if n > 1 else ""}\n(0°-{angles[n-1]:.0f}°)', fontsize=10)
    axes[idx].axis('off')

plt.suptitle('Unfiltered Back Projection: Accumulation Process',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("As more projections are added, the image becomes more recognizable.")
print("But the blurriness remains — this is inherent to unfiltered back projection.")

## 6. Quantitative Analysis: Error vs Number of Projections

Let's measure how the reconstruction error changes with the number of projections.

In [ ]:
# Error analysis
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.25, anti_aliasing=True)
sz = phantom.shape[0]

n_angles_range = np.arange(5, 365, 5)
errors_unfiltered = []
errors_filtered = []

for n in n_angles_range:
    angles = np.linspace(0, 180, n, endpoint=False)
    sinogram = radon(phantom, theta=angles, circle=True)
    
    # Unfiltered
    recon_uf = iradon(sinogram, theta=angles, filter_name=None, circle=True)
    recon_uf_norm = recon_uf * (phantom.max() / recon_uf.max()) if recon_uf.max() > 0 else recon_uf
    error_uf = np.sqrt(np.mean((phantom - recon_uf_norm)**2))
    errors_unfiltered.append(error_uf)
    
    # Filtered
    recon_f = iradon(sinogram, theta=angles, filter_name='ramp', circle=True)
    error_f = np.sqrt(np.mean((phantom - recon_f)**2))
    errors_filtered.append(error_f)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(n_angles_range, errors_unfiltered, 'r-', linewidth=2, label='Unfiltered BP')
axes[0].plot(n_angles_range, errors_filtered, 'b-', linewidth=2, label='Filtered BP')
axes[0].set_title('RMSE vs Number of Projections', fontsize=12)
axes[0].set_xlabel('Number of Projections')
axes[0].set_ylabel('RMSE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(n_angles_range, errors_unfiltered, 'r-', linewidth=2, label='Unfiltered BP')
axes[1].semilogy(n_angles_range, errors_filtered, 'b-', linewidth=2, label='Filtered BP')
axes[1].set_title('RMSE vs Projections (Log Scale)', fontsize=12)
axes[1].set_xlabel('Number of Projections')
axes[1].set_ylabel('RMSE (log scale)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Reconstruction Error: Unfiltered vs Filtered Back Projection',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Unfiltered BP error (180 proj): {errors_unfiltered[n_angles_range.tolist().index(180)]:.4f}")
print(f"Filtered BP error (180 proj):   {errors_filtered[n_angles_range.tolist().index(180)]:.4f}")
print("\nFiltered BP achieves MUCH lower error for the same number of projections!")

## Summary

What we learned:
1. **Back projection** "smears" each projection back along the original ray direction
2. **Simple (unfiltered) back projection** accumulates these smeared values: $\hat{f}(x,y) = \int g(x\cos\theta + y\sin\theta, \theta) \, d\theta$
3. This produces a **blurred** reconstruction due to the **$1/r$ point spread function**
4. More projections improve localization but **cannot remove the blur**
5. The blur corresponds to a **$1/|\omega|$** overweighting of low frequencies in the Fourier domain
6. **Filtered back projection** (next lesson) solves this by applying a ramp filter before back projecting